### CONFIGURACIÓN

In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from typing import List, Dict

/home/estudiante/tldr-uniandes/encoders-vs-decoders-classification/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Ingresar token de hugginface para subir a la plataforma] [Tambien poner las rutas correspondientes]

In [ ]:
MODEL_NAME: str = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit"
HF_DATASET_NAME: str = "andrewmos/indian-legal-summaries-chat-template"
HF_TOKEN: str = ""
df_j = pd.read_json("./just-nlp-folders/datasets/train/train_judg.jsonl", lines=True)
df_s = pd.read_json("./just-nlp-folders/datasets/train/train_ref_summ.jsonl", lines=True)

In [3]:
# CARGAR TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### CARGAR Y UNIR JSONL ORIGINALES

In [4]:
df = pd.merge(df_j, df_s, on="ID")

print("Total de registros originales:", len(df))

Total de registros originales: 1200


### CONSTRUIR messages ESTILO GEMMA-3

In [5]:
INSTRUCTION: str = (
    "Provide a concise and accurate summary of the following legal judgment. "
    "Focus on the key facts, the legal reasoning, and the final verdict."
)

def build_messages(full_judgment: str, summary: str) -> List[Dict[str, str]]:
    """
    Construye la estructura de mensajes para el fine-tuning de un modelo de chat.

    Esta función toma el texto completo de un juicio y su resumen correspondiente,
    y los formatea en una lista de diccionarios siguiendo el esquema de roles
    'user' y 'assistant'.

    Args:
        full_judgment (str): El texto completo del juicio legal.
        summary (str): El resumen del juicio (target).

    Returns:
        List[Dict[str, str]]: Una lista de mensajes donde cada mensaje es un diccionario
                              con las claves 'role' y 'content'.
    """
    user_msg = f"{INSTRUCTION}\n\n---\n\n{full_judgment}"
    return [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": summary}
    ]

df["messages"] = df.apply(
    lambda r: build_messages(r["Judgment"], r["Summary"]),
    axis=1
)

### SPLIT 80/20 (SIN CHUNKS)

In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    shuffle=True,
    random_state=42
)

print("Train:", len(train_df))
print("Test :", len(test_df))

Train: 960
Test : 240


### CONVERTIR A HUGGINGFACE DATASET

In [7]:
train_dataset = Dataset.from_pandas(
    train_df[["ID", "messages"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["ID", "messages"]],
    preserve_index=False
)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

### SUBIR AL HUGGINGFACE HUB

In [8]:
dataset_dict.push_to_hub(
    HF_DATASET_NAME,
    token=HF_TOKEN
)

print("📦 Nombre en HuggingFace:", HF_DATASET_NAME)


Uploading the dataset shards:   0%|                  | 0/1 [00:00<?, ? shards/s]


Creating parquet from Arrow format:   0%|                 | 0/1 [00:00<?, ?ba/s]


Creating parquet from Arrow format: 100%|█████████| 1/1 [00:00<00:00,  5.00ba/s]


Creating parquet from Arrow format: 100%|█████████| 1/1 [00:00<00:00,  4.95ba/s]


Processing Files (0 / 0): |                        |  0.00B /  0.00B            

...



New Data Upload: |                                 |  0.00B /  0.00B,  0.00B/s  



Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.16 shards/s]


Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.16 shards/s]

No files have been modified since last commit. Skipping to prevent empty commit.


📦 Nombre en HuggingFace: andrewmos/indian-legal-summaries-chat-template
